In [ ]:
# ===== SETUP: Mount Drive, clone repo, install deps, set paths =====
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/261-mini2'):
    !git clone https://github.com/gracee-chen/261-mini2.git /content/261-mini2
else:
    !cd /content/261-mini2 && git pull
%cd /content/261-mini2

!pip install -q segmentation-models-pytorch timm transformers scipy matplotlib
!pip install -q git+https://github.com/facebookresearch/sam2.git

DRIVE_DATA = '/content/drive/MyDrive/261-data'
DRIVE_CKPT = '/content/drive/MyDrive/261-checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/261-results'
VOC_ROOT = f'{DRIVE_DATA}/VOCtrainval_06-Nov-2007'
SAM2_CKPT = f'{DRIVE_DATA}/sam2.1_hiera_base_plus.pt'

for d in [DRIVE_DATA, DRIVE_CKPT, DRIVE_RESULTS]:
    os.makedirs(d, exist_ok=True)
for m in ['unet', 'deeplabv3plus', 'sam2', 'dinov2', 'dinov2_ft']:
    os.makedirs(f'{DRIVE_CKPT}/{m}', exist_ok=True)

!ln -sfn {DRIVE_CKPT} checkpoints
!ln -sfn {DRIVE_RESULTS} results
%env VOC_ROOT={VOC_ROOT}
%env SAM2_CKPT={SAM2_CKPT}
print('Setup done.')

In [ ]:
# ===== DOWNLOAD: VOC dataset + SAM2 weights (run once, skip if exists) =====
import os
voc_check = os.path.join(VOC_ROOT, 'VOCdevkit', 'VOC2007', 'JPEGImages')

if os.path.isdir(voc_check):
    print(f'VOC dataset already exists at {VOC_ROOT}')
else:
    from google.colab import files
    print('Upload your kaggle.json:')
    files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip install -q kaggle
    !kaggle datasets download -d zaraks/pascal-voc-2007 -p {DRIVE_DATA} --unzip
    if os.path.isdir(voc_check):
        print('VOC 2007 downloaded to Drive.')
    else:
        print('ERROR: download failed. Checking structure:')
        !ls -la {DRIVE_DATA}/

if not os.path.isfile(SAM2_CKPT):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt -O {SAM2_CKPT}
    print('SAM2 weights downloaded.')
else:
    print(f'SAM2 weights already at {SAM2_CKPT}')

In [ ]:
# ===== EXPLORE DATASET: sample images + class distribution =====
!python dataset/explore.py --root $VOC_ROOT --num-samples 3 --dist --save-dir results/explore

In [ ]:
# ===== RESULTS: dataset exploration samples =====
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('results/explore/*.png')):
    display(Image(filename=p, width=600))

In [ ]:
# ===== TRAIN: U-Net =====
!python train/train_unet.py --voc-root $VOC_ROOT --epochs 50 --checkpoint-dir checkpoints/unet

In [ ]:
# ===== TRAIN: DeepLabV3+ =====
!python train/train_deeplabv3plus.py --voc-root $VOC_ROOT --epochs 50 --checkpoint-dir checkpoints/deeplabv3plus

In [ ]:
# ===== TRAIN: DINOv2 Phase 1 (head only) + Phase 2 (full fine-tune) =====
!python train/train_dinov2.py --voc-root $VOC_ROOT --epochs 30 --lr 1e-3 --checkpoint-dir checkpoints/dinov2
!python train/train_dinov2.py --voc-root $VOC_ROOT --unfreeze-backbone --resume checkpoints/dinov2/best.pth --epochs 20 --lr 1e-5 --checkpoint-dir checkpoints/dinov2_ft

# Merge Phase 1 + Phase 2 training logs so compare.py reports total DINOv2 time
import json, os
p1 = 'checkpoints/dinov2/training_log.json'
p2 = 'checkpoints/dinov2_ft/training_log.json'
if os.path.isfile(p1) and os.path.isfile(p2):
    log1 = json.load(open(p1))
    log2 = json.load(open(p2))
    total_time = log1['total_time_sec'] + log2['total_time_sec']
    all_epochs = log1['epochs'] + [dict(e, epoch=e['epoch'] + log1['total_epochs']) for e in log2['epochs']]
    log2['total_time_sec'] = round(total_time, 1)
    log2['avg_epoch_time_sec'] = round(total_time / len(all_epochs), 1)
    log2['final_train_loss'] = all_epochs[-1]['train_loss']
    log2['final_val_loss'] = all_epochs[-1]['val_loss']
    log2['generalization_gap'] = round(all_epochs[-1]['val_loss'] - all_epochs[-1]['train_loss'], 6)
    log2['total_epochs'] = len(all_epochs)
    log2['epochs'] = all_epochs
    json.dump(log2, open(p2, 'w'), indent=2)
    print(f'Merged DINOv2 training logs: {log1["total_epochs"]}+{len(log2["epochs"])-log1["total_epochs"]} epochs, total {total_time/60:.1f} min')

In [ ]:
# ===== TRAIN: DINOv2 Phase 1 (head only) + Phase 2 (full fine-tune) =====
!python train/train_dinov2.py --voc-root $VOC_ROOT --epochs 30 --lr 1e-3 --checkpoint-dir checkpoints/dinov2
!python train/train_dinov2.py --voc-root $VOC_ROOT --unfreeze-backbone --resume checkpoints/dinov2/best.pth --epochs 20 --lr 1e-5 --checkpoint-dir checkpoints/dinov2_ft

In [ ]:
# ===== INFERENCE: all 4 models =====
!python inference/infer_unet.py --checkpoint checkpoints/unet/best.pth --voc-root $VOC_ROOT --num-samples 8 --output-dir results/inference/unet
!python inference/infer_deeplabv3plus.py --checkpoint checkpoints/deeplabv3plus/best.pth --voc-root $VOC_ROOT --num-samples 8 --output-dir results/inference/deeplabv3plus
!python inference/infer_sam2.py --checkpoint checkpoints/sam2/best.pth --sam2-ckpt $SAM2_CKPT --voc-root $VOC_ROOT --num-samples 8 --output-dir results/inference/sam2
!python inference/infer_dinov2.py --checkpoint checkpoints/dinov2_ft/best.pth --voc-root $VOC_ROOT --num-samples 8 --output-dir results/inference/dinov2

In [ ]:
# ===== EVALUATION: metrics + confusion matrix =====
!python evaluation/metrics/compute_metrics.py \
    --model-type unet deeplabv3plus sam2 dinov2 \
    --checkpoint checkpoints/unet/best.pth checkpoints/deeplabv3plus/best.pth checkpoints/sam2/best.pth checkpoints/dinov2_ft/best.pth \
    --voc-root $VOC_ROOT --sam2-ckpt $SAM2_CKPT --output-dir results/metrics

!python evaluation/metrics/confusion_matrix.py --model-type unet --checkpoint checkpoints/unet/best.pth --voc-root $VOC_ROOT --output-dir results/metrics

In [ ]:
# ===== ABLATION STUDIES (5 experiments, all U-Net based) =====
!python evaluation/ablation/ablation_backbone.py --voc-root $VOC_ROOT --epochs 25 --output-dir results/ablation
!python evaluation/ablation/ablation_augmentation.py --voc-root $VOC_ROOT --epochs 25 --output-dir results/ablation
!python evaluation/ablation/ablation_loss.py --voc-root $VOC_ROOT --epochs 25 --output-dir results/ablation
!python evaluation/ablation/ablation_pretrain.py --voc-root $VOC_ROOT --epochs 25 --output-dir results/ablation
!python evaluation/ablation/ablation_resolution.py --voc-root $VOC_ROOT --epochs 25 --batch-size 4 --output-dir results/ablation

In [ ]:
# ===== VISUALIZATION: mosaic + top3/worst3 =====
!python evaluation/visualization/visualize_mosaic.py \
    --voc-root $VOC_ROOT \
    --models unet:checkpoints/unet/best.pth deeplabv3plus:checkpoints/deeplabv3plus/best.pth dinov2:checkpoints/dinov2_ft/best.pth \
    --num-images 6 --output results/visualization/mosaic.png

!python evaluation/visualization/visualize_comparison.py \
    --voc-root $VOC_ROOT --model-type unet deeplabv3plus \
    --checkpoint checkpoints/unet/best.pth checkpoints/deeplabv3plus/best.pth \
    --metric miou --output-dir results/visualization

!python evaluation/visualization/visualize_comparison.py \
    --voc-root $VOC_ROOT --model-type unet deeplabv3plus \
    --checkpoint checkpoints/unet/best.pth checkpoints/deeplabv3plus/best.pth \
    --metric person --output-dir results/visualization

In [ ]:
# ===== CROSS-MODEL COMPARISON =====
!python evaluation/compare.py \
    --models unet deeplabv3plus sam2 dinov2 \
    --checkpoint-dirs checkpoints/unet checkpoints/deeplabv3plus checkpoints/sam2 checkpoints/dinov2_ft \
    --metrics-dir results/metrics --output-dir results/compare

In [ ]:
# ===== RESULTS: inference/unet =====
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('results/inference/unet/*.png')):
    display(Image(filename=p, width=600))

In [ ]:
# ===== RESULTS: inference/deeplabv3plus =====
for p in sorted(glob.glob('results/inference/deeplabv3plus/*.png')):
    display(Image(filename=p, width=600))

In [ ]:
# ===== RESULTS: inference/sam2 =====
for p in sorted(glob.glob('results/inference/sam2/*.png')):
    display(Image(filename=p, width=600))

In [ ]:
# ===== RESULTS: inference/dinov2 =====
for p in sorted(glob.glob('results/inference/dinov2/*.png')):
    display(Image(filename=p, width=600))

In [ ]:
# ===== RESULTS: confusion matrix =====
display(Image(filename='results/metrics/unet_confusion_matrix.png', width=700))

In [ ]:
# ===== RESULTS: loss curves =====
display(Image(filename='results/compare/loss_curves.png', width=800))

In [ ]:
# ===== RESULTS: metrics bar chart =====
display(Image(filename='results/compare/metrics_bar.png', width=800))

In [ ]:
# ===== RESULTS: training time comparison =====
display(Image(filename='results/compare/training_time_bar.png', width=700))

In [ ]:
# ===== RESULTS: generalization gap =====
display(Image(filename='results/compare/generalization_bar.png', width=700))

In [ ]:
# ===== RESULTS: mosaic visualization =====
display(Image(filename='results/visualization/mosaic.png', width=900))

In [ ]:
# ===== RESULTS: top-3 best (mIoU) =====
display(Image(filename='results/visualization/top3_miou.png', width=800))

In [ ]:
# ===== RESULTS: worst-3 (mIoU) =====
display(Image(filename='results/visualization/worst3_miou.png', width=800))

In [ ]:
# ===== RESULTS: top-3 best (person class) =====
display(Image(filename='results/visualization/top3_person.png', width=800))

In [ ]:
# ===== RESULTS: worst-3 (person class) =====
display(Image(filename='results/visualization/worst3_person.png', width=800))

In [ ]:
# ===== RESULTS: summary table =====
with open('results/compare/summary_table.txt') as f:
    print(f.read())